# Inventário do STEP `#F-M-1219` (LD Celulose — O₃)
Lê o STEP **com os nomes das peças** e gera um CSV pequeno (nome, posição, maior dimensão, nº de faces).
É esse CSV que você manda de volta — ele identifica exaustores, misturadores e tubulação de saída de O₃.

**Rode as células em ordem** (Ambiente de execução → Executar tudo).

In [ ]:
# 1) Motor CAD (OpenCASCADE) + leitor de RAR — ~30 s
!pip install -q cadquery-ocp rarfile
!apt-get -qq install -y unrar > /dev/null 2>&1 || true

## 2) Enviar o arquivo
Envie o **`#F-M-1219_copy.rar` (45 MB)** — é 13× menor que o .stp de 600 MB e sobe muito mais rápido.
Também aceita .zip ou o .stp direto. Vai abrir um botão **Escolher arquivos** abaixo da célula.

In [ ]:
import os, glob, rarfile, zipfile
from google.colab import files
enviados = files.upload()
nome = list(enviados.keys())[0]
os.makedirs('/content/step', exist_ok=True)
if nome.lower().endswith('.rar'):
    rarfile.RarFile(nome).extractall('/content/step')
elif nome.lower().endswith('.zip'):
    zipfile.ZipFile(nome).extractall('/content/step')
else:
    os.replace(nome, '/content/step/' + os.path.basename(nome))
achados = [f for f in glob.glob('/content/step/**/*', recursive=True) if f.lower().endswith(('.stp', '.step'))]
ARQ_STEP = max(achados, key=os.path.getsize)
print('STEP:', ARQ_STEP, f'{os.path.getsize(ARQ_STEP)/1e6:.0f} MB')

## 3) Ler o STEP com nomes e gerar o inventário
Pode levar **5 a 15 min** para 600 MB. Se o Colab reiniciar por falta de memória, use *Ambiente de execução → Alterar tipo → RAM alta* (se disponível) ou exporte do Solid Edge só as peças visíveis.
No fim, o CSV baixa sozinho — **mande esse CSV no chat**.

In [ ]:
import os, csv, zipfile, time
from google.colab import files
from OCP.STEPCAFControl import STEPCAFControl_Reader
from OCP.TDocStd import TDocStd_Document
from OCP.TCollection import TCollection_ExtendedString
from OCP.XCAFDoc import XCAFDoc_DocumentTool
from OCP.TDF import TDF_LabelSequence, TDF_Label
from OCP.TDataStd import TDataStd_Name
from OCP.TopLoc import TopLoc_Location
from OCP.Bnd import Bnd_Box
from OCP.BRepBndLib import BRepBndLib
from OCP.TopExp import TopExp_Explorer
from OCP.TopAbs import TopAbs_FACE, TopAbs_SOLID
from OCP.IFSelect import IFSelect_RetDone

ARQUIVO = ARQ_STEP                          # definido na célula 2
SAIDA   = "/content/inventario_F-M-1219.csv"

t0 = time.time()
doc = TDocStd_Document(TCollection_ExtendedString("doc"))
r = STEPCAFControl_Reader(); r.SetNameMode(True)
assert r.ReadFile(ARQUIVO) == IFSelect_RetDone, "falha ao ler o STEP"
r.Transfer(doc)
print(f"lido em {time.time()-t0:.0f} s")
st = XCAFDoc_DocumentTool.ShapeTool_s(doc.Main())

def nome(lab):
    a = TDataStd_Name()
    return a.Get().ToExtString() if lab.FindAttribute(TDataStd_Name.GetID_s(), a) else "?"

linhas = []
def visita(lab, loc, caminho):
    if st.IsReference_s(lab):
        ref = TDF_Label(); st.GetReferredShape_s(lab, ref)
        visita(ref, loc.Multiplied(st.GetLocation_s(lab)), caminho + [nome(lab)]); return
    if st.IsAssembly_s(lab):
        f = TDF_LabelSequence(); st.GetComponents_s(lab, f, False)
        for i in range(1, f.Length() + 1): visita(f.Value(i), loc, caminho)
        return
    shp = st.GetShape_s(lab).Moved(loc)
    b = Bnd_Box(); BRepBndLib.Add_s(shp, b, False)
    if b.IsVoid(): return
    x0, y0, z0, x1, y1, z1 = (v / 1000 for v in b.Get())      # mm -> m
    nf = 0; e = TopExp_Explorer(shp, TopAbs_FACE)
    while e.More(): nf += 1; e.Next()
    ns = 0; e = TopExp_Explorer(shp, TopAbs_SOLID)
    while e.More(): ns += 1; e.Next()
    dmax = max(x1 - x0, y1 - y0, z1 - z0)
    linhas.append([" / ".join(caminho[-3:]), nome(lab), round(x0, 3), round(y0, 3), round(z0, 3),
                   round(x1, 3), round(y1, 3), round(z1, 3), round(dmax, 3), nf, ns])

top = TDF_LabelSequence(); st.GetFreeShapes(top)
for i in range(1, top.Length() + 1): visita(top.Value(i), TopLoc_Location(), [])
with open(SAIDA, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["caminho", "peca", "xmin", "ymin", "zmin", "xmax", "ymax", "zmax", "maior_dim_m", "faces", "solidos"])
    w.writerows(linhas)
print(len(linhas), "peças ->", SAIDA)
files.download(SAIDA)


## 4) (opcional) Olhada rápida no inventário

In [ ]:
import pandas as pd
df = pd.read_csv(SAIDA)
print(len(df), 'peças')
for chave in ['24-9', 'FAN', 'EXAUST', 'VENT', 'O3', 'OZ', 'MIX', 'DUTO', 'DUCT']:
    m = df[df['peca'].str.contains(chave, case=False, na=False) | df['caminho'].str.contains(chave, case=False, na=False)]
    if len(m): print(f'\n=== {chave}: {len(m)} peças'); print(m[['peca','maior_dim_m','zmin','zmax']].head(15).to_string(index=False))